<div style='background:#1a3a5c;color:white;padding:22px 30px;border-radius:10px;font-family:sans-serif'>
<h1 style='margin:0 0 6px 0;font-size:1.6em'>⚡ DESAFÍO RELÁMPAGO — Sesión 04</h1>
<h2 style='margin:0 0 10px 0;font-weight:300;font-size:1.1em'>El Eco como Instrumento: Buffer Circular, Feedback y Saturación</h2>
<p style='margin:0;opacity:0.8;font-size:0.95em'>
Tipo: <strong>Predecir → Reparar → Construir</strong><br>
Objetivo: Entender el buffer circular reparando bugs, explorar cómo el feedback
transforma un simple eco en un instrumento dub, y descubrir por qué <code>tanh</code>
es el ingrediente secreto que permite romper las reglas.
</p></div>

## 0 · Setup y Señal de Prueba

Usaremos un **golpe percusivo corto** (como un rim shot) seguido de silencio.
Es la señal ideal para estudiar delays: cada eco aparece como una copia separada y visible.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import wave, struct, os

FS = 44100

def save_wav(filename, audio, fs=FS):
    audio = np.array(audio, dtype=np.float32)
    peak = np.max(np.abs(audio))
    if peak > 0: audio = audio / peak * 0.88
    raw = audio.tobytes()
    with wave.open(filename, 'w') as wf:
        wf.setnchannels(1); wf.setsampwidth(4); wf.setframerate(fs)
        wf.writeframes(raw)
    print(f'  Guardado: {filename}  ({os.path.getsize(filename)/1024:.0f} KB)')

# Señal de prueba: golpe percusivo corto (50 ms de ruido con decaimiento exponencial)
duration = 3.0  # segundos totales
N = int(duration * FS)
pulse = np.zeros(N, dtype=np.float32)
hit_len = int(0.05 * FS)  # 50 ms
pulse[:hit_len] = np.random.randn(hit_len) * np.exp(-np.linspace(0, 8, hit_len))
pulse = pulse.astype(np.float32)

save_wav('dr04_input.wav', pulse)

fig, ax = plt.subplots(figsize=(11, 2.5))
t = np.arange(N) / FS
ax.plot(t, pulse, color='steelblue', lw=0.5)
ax.set(xlabel='Tiempo (s)', title='Señal de prueba: golpe percusivo + silencio')
ax.axvspan(0, 0.05, alpha=0.15, color='orange', label='Golpe (50 ms)')
ax.legend(); plt.tight_layout(); plt.show()
print(f'Setup OK — FS = {FS} Hz, duración = {duration} s')

## 1 · Predecir en Papel: ¿Qué hace este código?

<div style='background:#fff3cd;border-left:5px solid #ffc107;padding:16px 20px;border-radius:6px;font-family:sans-serif'>
<h3 style='margin:0 0 10px 0'>🔴 PAUSA — No ejecutes todavía</h3>

Lee este código con tu compañero(a) y respondan **en papel**:

```python
buffer = np.zeros(11025)   # ¿cuántos ms representa esto a 44100 Hz?
write_ptr = 0

def process_sample(x_in, feedback=0.6):
    read_ptr = (write_ptr - 11025) % 11025
    y_out = x_in + feedback * buffer[read_ptr]
    buffer[write_ptr] = y_out
    write_ptr = (write_ptr + 1) % 11025
    return y_out
```

**Preguntas:**
1. ¿Cuántos milisegundos de delay produce este buffer? *(Pista: muestras / fs × 1000)*
2. Si `feedback = 0.6`, ¿cuál es la amplitud del 1er eco? ¿Y del 3er eco? *(Pista: g^n)*
3. ¿Cuántos ecos se escucharían antes de ser inaudibles? *(Criterio: g^n < 0.01)*
4. **Este código tiene un BUG de Python.** ¿Cuál es? *(Pista: scope de variables)*

✏️ *Escribe tus respuestas antes de continuar.*
</div>

### 1.1 · Respuestas

| Pregunta | Respuesta |
|----------|-----------|
| Delay en ms | 11025 / 44100 × 1000 = **250 ms** (un cuarto de segundo) |
| Amplitud 1er eco | 0.6¹ = **0.6** |
| Amplitud 3er eco | 0.6³ = **0.216** |
| Ecos audibles | 0.6ⁿ < 0.01 → n > log(0.01)/log(0.6) ≈ **9 ecos** |
| Bug de Python | `write_ptr` es variable global. Dentro de una función, Python la trata como local al asignarla. Necesita `global write_ptr` o (mejor) usar una clase. |

## 2 · Construir: De la Ecuación al Buffer Circular

La ecuación del delay con feedback es sorprendentemente simple:

$$y[n] = x[n] + g \cdot y[n - D]$$

Donde:
- $x[n]$ = entrada (la señal seca)
- $y[n-D]$ = lo que salió hace $D$ muestras (el eco)
- $g$ = feedback (cuánto del eco se re-inyecta)

Vamos a implementarla en **3 pasos**, cada uno más eficiente que el anterior.

### Paso 2A — La versión "ingenua" (pero correcta)

<div style='background:#d4edda;border-left:5px solid #28a745;padding:16px 20px;border-radius:6px;font-family:sans-serif'>
<h3 style='margin:0 0 8px 0'>✏️ MANOS A LA OBRA</h3>

Implementa la ecuación <strong>directamente</strong> usando un array <code>y</code> largo.
No te preocupes por eficiencia — solo por que suene correcto.

```
y = np.zeros(len(x) + D)
for n in range(len(x)):
    y[n] = x[n] + g * (y[n - D] if n >= D else 0)
```

1. Calcula `D` para un delay de 250 ms a 44100 Hz.
2. Aplícalo a la señal `pulse` con `g = 0.6`.
3. Grafícalo y verifica: ¿ves ecos separados cada 250 ms?
</div>

In [ ]:
# ✅ PASO 2A — Delay con array largo (versión ingenua)
def delay_naive(x, delay_ms=250, feedback=0.6, fs=FS):
    D = int(delay_ms * fs / 1000)
    y = np.zeros(len(x) + D)
    for n in range(len(x)):
        y[n] = x[n] + feedback * (y[n - D] if n >= D else 0)
    return y[:len(x)]  # recortar al largo original

out_naive = delay_naive(pulse, delay_ms=250, feedback=0.6)
save_wav('dr04_naive.wav', out_naive)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, out_naive, color='steelblue', lw=0.5)
for echo_n in range(1, 10):
    echo_time = echo_n * 0.250
    if echo_time < duration:
        ax.axvline(echo_time, color='red', alpha=0.3, lw=0.8)
ax.set(xlabel='Tiempo (s)', title='Paso 2A: delay "ingenuo" — ¡funciona!')
plt.tight_layout(); plt.show()
print(f'D = {int(250 * FS / 1000)} muestras = 250 ms')
print(f'Ecos visibles cada 250 ms con amplitud decreciente: 0.6, 0.36, 0.216...')

### Paso 2B — ¿Por qué no podemos usar esto en vivo?

<div style='background:#fff3cd;border-left:5px solid #ffc107;padding:16px 20px;border-radius:6px;font-family:sans-serif'>
<h3 style='margin:0 0 10px 0'>🔴 PREGUNTA: El Problema de la Memoria</h3>

La versión 2A funciona perfecto para procesar un archivo grabado. Pero imagina que estás tocando guitarra **en vivo** con un pedal de delay:

1. ¿Cuánta RAM necesita el array `y` para 5 minutos de audio a 44100 Hz? *(Pista: `5 × 60 × 44100 × 4 bytes`)*
2. Cuando empieza la canción, ¿ya sabes cuánto va a durar? ¿Puedes crear el array por adelantado?
3. Para un delay de 250 ms, ¿realmente necesitas recordar los **5 minutos completos** o solo los últimos **250 ms**?

El **buffer circular** resuelve esto: usa un array de tamaño fijo (solo `D` muestras) y recicla posiciones. Como un cassette de cinta que da vueltas.
</div>

### Paso 2C — El Buffer Circular: misma ecuación, memoria fija

<div style='background:#d4edda;border-left:5px solid #28a745;padding:16px 20px;border-radius:6px;font-family:sans-serif'>
<h3 style='margin:0 0 8px 0'>✏️ MANOS A LA OBRA</h3>

Ahora transforma tu delay en una **clase** que procese **una muestra a la vez** usando un buffer circular de tamaño fijo `D`.

La idea:
- Un array `buffer` de `D` posiciones (inicializado en ceros).
- Un puntero `write_ptr` que avanza y **da la vuelta** con módulo `D`.
- **Leer antes de escribir:** en la posición `write_ptr` está lo que se escribió hace exactamente `D` muestras — léelo, úsalo como eco, y luego sobrescribe con el valor nuevo.

Ventaja: procesa **una muestra a la vez** (sirve para tiempo real) y usa solo `D` posiciones de memoria en vez de millones.
</div>

In [ ]:
# ✅ PASO 2C — Delay con buffer circular (clase)
class DelayLine:
    """Buffer circular para delay con feedback — procesa una muestra a la vez."""
    def __init__(self, delay_ms, fs=FS):
        self.N = int(delay_ms * fs / 1000)
        self.buffer = np.zeros(self.N)
        self.write_ptr = 0

    def process(self, x_in, feedback=0.6):
        # Leer: en write_ptr está lo que se escribió hace N muestras
        y_delayed = self.buffer[self.write_ptr]
        # Ecuación del delay (¡la misma que en 2A!)
        y_out = x_in + feedback * y_delayed
        # Escribir: sobrescribir con el valor nuevo
        self.buffer[self.write_ptr] = y_out
        # Avanzar puntero circular
        self.write_ptr = (self.write_ptr + 1) % self.N
        return y_out

# Verificar que suena IGUAL que la versión ingenua
dl = DelayLine(250)
out_circular = np.array([dl.process(pulse[n], feedback=0.6) for n in range(len(pulse))])
save_wav('dr04_circular.wav', out_circular)

# Comparar numéricamente
diff = np.max(np.abs(out_naive - out_circular))
print(f'Diferencia máxima entre versión ingenua y buffer circular: {diff:.2e}')
print(f'Memoria versión ingenua:   {(len(pulse) + int(250*FS/1000)) * 4 / 1024:.0f} KB')
print(f'Memoria buffer circular:   {int(250*FS/1000) * 4 / 1024:.0f} KB')
print(f'¡Misma ecuación, {(len(pulse) + int(250*FS/1000)) // int(250*FS/1000):.0f}× menos memoria!')

## 3 · Explorar: La Anatomía del Feedback

Ahora que el delay funciona, vamos a **escuchar** qué pasa con distintos valores de feedback.

La ecuación de cada eco es: `amplitud del eco n = feedback^n`

| feedback | Eco 1 | Eco 3 | Eco 5 | Comportamiento |
|----------|-------|-------|-------|----------------|
| 0.3 | 0.30 | 0.03 | 0.002 | Eco sutil, desaparece rápido |
| 0.6 | 0.60 | 0.22 | 0.078 | Eco presente, musical |
| 0.85 | 0.85 | 0.61 | 0.44 | Eco largo, estilo dub |
| 0.95 | 0.95 | 0.86 | 0.77 | Casi infinito, hipnótico |
| 1.2 | 1.20 | 1.73 | 2.49 | **CRECE** → explota |

In [ ]:
# Generar y graficar 4 variantes de feedback (sin saturación)
fb_values = [0.3, 0.6, 0.85, 0.95]
results = {}

fig, axes = plt.subplots(len(fb_values), 1, figsize=(12, 2.2 * len(fb_values)), sharex=True)

for i, fb in enumerate(fb_values):
    dl = DelayLine(250)
    out = np.array([dl.process(pulse[n], feedback=fb) for n in range(len(pulse))])
    results[fb] = out
    save_wav(f'dr04_fb{int(fb*100):03d}.wav', out)

    axes[i].plot(t, out, color='steelblue', lw=0.5)
    axes[i].set_ylabel(f'fb={fb}')
    # Marcar los ecos con líneas verticales
    for echo_n in range(1, 10):
        echo_time = echo_n * 0.250
        if echo_time < duration:
            axes[i].axvline(echo_time, color='red', alpha=0.3, lw=0.8)

axes[-1].set_xlabel('Tiempo (s)')
axes[0].set_title('Comparación de feedback: cada línea roja = un eco')
plt.tight_layout(); plt.show()

<div style='background:#fff3cd;border-left:5px solid #ffc107;padding:16px 20px;border-radius:6px;font-family:sans-serif'>
<h3 style='margin:0 0 10px 0'>🔴 PREGUNTA: El Límite de la Estabilidad</h3>

Observa las formas de onda:
1. ¿En cuál de los 4 gráficos los ecos **todavía no han desaparecido** a los 3 segundos?
2. ¿Qué pasaría si pusieras `feedback = 1.0` exactamente? *(Pista: 1.0^n = ?)*
3. ¿Y si pusieras `feedback = 1.2`? *(Pista: 1.2^10 ≈ 6.2)*

La regla de oro de los sistemas LTI: **|feedback| < 1 → estable, |feedback| ≥ 1 → inestable.**

Pero en el dub jamaicano, los ingenieros de sonido **querían** llevar el feedback al límite...
¿Cómo lo hacían sin destruir los parlantes?
</div>

## 4 · El Truco Dub: Saturación con `tanh`

En el mundo analógico, la cinta magnética y los amplificadores a tubos **saturan naturalmente**:
cuando la señal es muy grande, el hardware la comprime suavemente hacia un límite.

La función `tanh(x)` replica ese comportamiento:
- Si `|x| < 0.5` → `tanh(x) ≈ x` (transparente, no altera la señal)
- Si `|x| → grande` → `tanh(x) → ±1` (comprime, nunca explota)

El truco: ponemos `tanh` **dentro del loop de feedback**, no después de la salida.
Así el feedback se auto-regula en cada vuelta.

In [ ]:
# Visualizar tanh: la saturación suave
x_range = np.linspace(-4, 4, 500)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(x_range, x_range, '--', color='gray', alpha=0.5, label='Lineal (sin saturación)')
ax.plot(x_range, np.tanh(x_range), color='crimson', lw=2.5, label='tanh(x)')
ax.plot(x_range, np.clip(x_range, -1, 1), ':', color='navy', lw=1.5, label='clip (hard clipping)')
ax.axhline(1, color='gray', alpha=0.3); ax.axhline(-1, color='gray', alpha=0.3)
ax.set(xlabel='x (amplitud de entrada)', ylabel='Salida', title='Saturación suave vs dura')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print('tanh es más musical que clip: la transición es gradual, no abrupta.')

<div style='background:#d4edda;border-left:5px solid #28a745;padding:16px 20px;border-radius:6px;font-family:sans-serif'>
<h3 style='margin:0 0 8px 0'>✏️ TU TURNO — Agrega tanh al DelayLine</h3>

Modifica tu clase `DelayLine` para que acepte un parámetro `saturate=True`.
Cuando esté activado, lo que se **escribe** en el buffer debe pasar por `np.tanh()`.

**Importante:** la saturación va en lo que se **almacena** (el feedback loop), NO en la salida `y_out`.
¿Por qué? Porque queremos que `y_out` sea la señal limpia + eco. La distorsión solo debe afectar
a las copias sucesivas del eco, no a la señal directa.

```
y_out = x_in + feedback * y_delayed          # salida limpia
buffer[write_ptr] = np.tanh(y_out)           # lo que se guarda se satura
```
</div>

In [ ]:
# ✅ SOLUCIÓN: DelayLine con saturación
class DelayLineSat:
    """Buffer circular con saturación opcional en el feedback loop."""
    def __init__(self, delay_ms, fs=FS):
        self.N = int(delay_ms * fs / 1000)
        self.buffer = np.zeros(self.N)
        self.write_ptr = 0

    def process(self, x_in, feedback=0.6, saturate=False):
        y_delayed = self.buffer[self.write_ptr]
        y_out = x_in + feedback * y_delayed

        # Lo que se guarda en el buffer (el feedback loop)
        if saturate:
            self.buffer[self.write_ptr] = np.tanh(y_out)
        else:
            self.buffer[self.write_ptr] = y_out

        self.write_ptr = (self.write_ptr + 1) % self.N
        return y_out

# Comparación dramática: fb=1.2 SIN vs CON saturación
print('=== feedback = 1.2 ===')

# Sin saturación (explota)
dl_boom = DelayLineSat(250)
out_boom = np.array([dl_boom.process(pulse[n], 1.2, saturate=False) for n in range(len(pulse))])
print(f'Sin saturación: max = {np.max(np.abs(out_boom)):.1f}  (¡EXPLOTA!)')

# Con saturación (estable y musical)
dl_sat = DelayLineSat(250)
out_sat = np.array([dl_sat.process(pulse[n], 1.2, saturate=True) for n in range(len(pulse))])
print(f'Con tanh:        max = {np.max(np.abs(out_sat)):.3f}  (estable)')
save_wav('dr04_fb120_saturated.wav', out_sat)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t, np.clip(out_boom, -5, 5), color='red', lw=0.5)
axes[0].set(ylabel='Amplitud', title='feedback=1.2 SIN saturación (clipped a ±5 para visualizar)')
axes[0].axhline(1, color='gray', ls='--', alpha=0.3); axes[0].axhline(-1, color='gray', ls='--', alpha=0.3)
axes[1].plot(t, out_sat, color='darkgreen', lw=0.5)
axes[1].set(xlabel='Tiempo (s)', ylabel='Amplitud', title='feedback=1.2 CON tanh — auto-oscilación controlada')
axes[1].axhline(1, color='gray', ls='--', alpha=0.3); axes[1].axhline(-1, color='gray', ls='--', alpha=0.3)
plt.tight_layout(); plt.show()
print('El eco con tanh entra en auto-oscilación pero NUNCA explota. Ese es el sonido dub.')

## 5 · Desafío Final: El Mini Spring Reverb

Un **spring reverb** (reverb de resorte) tiene un sonido metálico muy particular.
Su secreto: son **múltiples delays cortos** con tiempos que **no son múltiplos entre sí**.

Si los tiempos fueran múltiplos (ej: 10, 20, 30 ms), los ecos se alinean y suenan como un eco con pitch.
Si los tiempos son primos o inarmónicos (ej: 11, 17, 23 ms), los ecos se dispersan y crean una textura difusa.

<div style='background:#e8f4f8;border-left:4px solid #17a2b8;padding:12px 16px;border-radius:4px'>

**Tu misión:** Implementar 3 delays en paralelo con tiempos `[11, 17, 23]` ms y
feedback ≈ 0.85. Mezclar sus salidas sumándolas (dividiendo por 3 para normalizar).

Compara:
- **Spring:** delays `[11, 17, 23]` ms → coloración metálica
- **Eco simple:** un solo delay de 250 ms → repetición clara

Esta variante "spring reverb" es exactamente lo que la Fun Task 04 les pide como entregable extra.
</div>

In [ ]:
# ✅ SOLUCIÓN: Spring Reverb con 3 delays paralelos
class SpringReverb:
    """Múltiples delays cortos en paralelo → textura de reverb de resorte."""
    def __init__(self, delay_times_ms, fs=FS):
        self.delays = [DelayLineSat(d, fs) for d in delay_times_ms]
        self.n_delays = len(self.delays)

    def process(self, x_in, feedback=0.85):
        y_out = x_in
        for dl in self.delays:
            y_out += dl.process(x_in, feedback=feedback, saturate=True) / self.n_delays
        return y_out

# Generar spring reverb
spring = SpringReverb([11, 17, 23])
out_spring = np.array([spring.process(pulse[n], feedback=0.85) for n in range(len(pulse))])
save_wav('dr04_spring_reverb.wav', out_spring)

# Comparar visualmente con eco simple
dl_simple = DelayLineSat(250)
out_simple = np.array([dl_simple.process(pulse[n], 0.6, saturate=False) for n in range(len(pulse))])

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t, out_simple, color='steelblue', lw=0.5)
axes[0].set(ylabel='Amplitud', title='Eco simple: 250 ms, fb=0.6 — ecos separados y claros')
axes[1].plot(t, out_spring, color='darkorange', lw=0.5)
axes[1].set(xlabel='Tiempo (s)', ylabel='Amplitud',
            title='Spring reverb: [11, 17, 23] ms, fb=0.85 — textura difusa y metálica')
plt.tight_layout(); plt.show()
print('¿Escuchas la diferencia? El spring reverb tiene una "cola" continua en vez de ecos separados.')

## 6 · Exploración Libre: Tus Parámetros

<div style='background:#e8f4f8;border-left:4px solid #17a2b8;padding:12px 16px;border-radius:4px'>

Experimenta cambiando valores y escucha:

| Parámetro | Valor bajo | Valor alto | Efecto |
|-----------|-----------|-----------|--------|
| `delay_ms` | 10–30 ms | 200–500 ms | Corto = coloración/flanger · Largo = eco claro |
| `feedback` | 0.3 | 0.95 | Bajo = eco sutil · Alto = eco infinito |
| `feedback` | 1.0–1.5 (con tanh) | — | Auto-oscilación controlada, sonido dub |
| Spring `delay_times` | `[11,17,23]` | `[37,59,83]` | Más corto = más metálico · Más largo = más "room" |

**Conexión con la Fun Task 04:**  
Todo lo que construiste aquí es exactamente lo que necesitas para la Dub Machine.
La tarea te pide generar 4 archivos con fb distintos + el spring reverb.
Ya tienes el código base — ahora solo falta aplicarlo a una señal más interesante.
</div>

In [ ]:
# Espacio para experimentar — modifica y ejecuta
DELAY_MS  = 250
FEEDBACK  = 0.85
SATURATE  = True

dl_exp = DelayLineSat(DELAY_MS)
out_exp = np.array([dl_exp.process(pulse[n], FEEDBACK, SATURATE) for n in range(len(pulse))])
save_wav(f'dr04_experimento.wav', out_exp)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, out_exp, color='purple', lw=0.5)
ax.set(xlabel='Tiempo (s)', ylabel='Amplitud',
       title=f'Tu experimento: delay={DELAY_MS}ms, fb={FEEDBACK}, saturate={SATURATE}')
plt.tight_layout(); plt.show()

## Resumen

| Concepto | Lo que aprendiste | Conexión con Fun Task 04 |
|----------|-------------------|--------------------------|
| **Buffer circular** | Clase con `write_ptr`, lectura antes de escritura | Base de tu `delay_fb()` |
| **Feedback < 1** | Ecos decaen: amplitud = fb^n | Archivos `fb05`, `fb085`, `fb095` |
| **Feedback ≥ 1** | Sin saturación → explota | Entender *por qué* necesitas saturar |
| **tanh en el loop** | Permite fb > 1 de forma estable | Archivo `fb12_saturated` |
| **Spring reverb** | Múltiples delays cortos no-armónicos | Archivo `spring.wav` |

**Regla de oro:** El saturador va **dentro** del loop de feedback, no después de la salida.

**Próximo paso:** En la Fun Task 04, usarás `np.clip(y, -1, 1)` como saturador (más simple que tanh pero mismo principio). Aplícalo a una señal real (guitarra, drum, voz) y explora sistemáticamente los 4 valores de feedback.